In [1]:
import numpy as np
import pandas as pd
import os
import random
import re


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns



import ete3 as ete

import scipy.stats as stats
from functools import *

import Bio
from Bio import Entrez
from Bio import SeqIO
from Bio import Seq
from Bio.SeqUtils import GC


sns.set_context("paper")
%matplotlib inline



In [2]:
#Specify translation table
#more info: https://www.ncbi.nlm.nih.gov/Taxonomy/Utils/wprintgc.cgi#SG1



def find_orfs_with_trans(seq, start_codons=['ATG', 'CTG'], stop_codons = ['TAA','TAG', 'TGA'],
                        min_len  = (8*3)):
    
    '''
    Function that finds all ORFs in the 6 possible reading frames of a sequence. 
    The function uses the standard codon table.
    Can modify start codons to include more alternative start codons. 
    'CTG' start codon is still encoded as Met
    Returns a list of tuples for all ORFs identified.
    Tuple is in the following order:
        (start, end, strand, seq_nt, seq_aa)
        
    For now, skip ambigous sequences
    min_len corresponds to bp including start and stop codon

    '''
    
#     trans_table = Bio.Data.CodonTable.CodonTable(nucleotide_alphabet=Bio.Data.CodonTable.standard_dna_table.nucleotide_alphabet, 
#                                                  protein_alphabet=Bio.Data.CodonTable.standard_dna_table.protein_alphabet,
#                                                  forward_table=Bio.Data.CodonTable.standard_dna_table.forward_table,
#                                                  back_table=Bio.Data.CodonTable.standard_dna_table.back_table,
#                                                  start_codons=start_codons,
#                                                  stop_codons=Bio.Data.CodonTable.standard_dna_table.stop_codons
#                                                 )
    #trans_table =Bio.Data.CodonTable.standard_dna_table
    
    answer = []
    seq_len = len(seq)
    
    start_codons=['ATG', 'CTG']
    stop_codons = ['TAA','TAG', 'TGA']
    #parse start and stop codons:
    start_codons = [f"{codon}" for codon in start_codons]
    start_code= '|'.join(start_codons)
    stop_codons = [f"{codon}" for codon in stop_codons]
    stop_code = '|'.join(stop_codons)

    #Define string pattern for ORF 
    pattern = re.compile(f'(?=(({start_code})(?:...)*?)(?=({stop_code})))')

    
    df = {'start': [],
         'end': [],
          'strand': [],
         'seq_nt': [],
         'seq_aa': [], 
         'length_aa': []}
    
    #loop through strands    
    for strand, nuc in [(+1, seq), (-1, seq.reverse_complement())]:
        
        #find orfs
        orfs = [o[0]+o[2] for o in pattern.findall(str(nuc)) if ((len(o[0]) >= min_len) & (len(o[0]) % 3 == 0))]
        starts = [nuc.find(o) for o in orfs]
        ends = [starts[i] + len(o) for i, o in enumerate(orfs)]
        seqs_nt = [str(nuc[starts[i]:ends[i]]) for i,o in enumerate(orfs)] 
        seqs_aa = []
        lengths_aa = []

        for seq_nt in seqs_nt:
            try:
                if seq_nt[0:3] == 'CTG':
                    seq_nt_trans = 'ATG' + seq_nt[3:]
                else:
                    seq_nt_trans = seq_nt
                    
                
                s = str(Seq.Seq(seq_nt_trans).translate())[0:-1]
                seqs_aa.append(s)
                lengths_aa.append(len(s))
        
            except Bio.Data.CodonTable.TranslationError:
                print('Ambiguous Codon Detected for seq:')
                print(seq_nt)
                s = 'Ambiguous Codon Detected'
                seqs_aa.append(s)
                lengths_aa.append(0)
                
        #modify to reverse complement sequence postion 
        if strand == -1:
            starts = [len(seq)+1-s for s in starts]
            ends = [len(seq)+1-e for e in ends]
            
        df['start'] =df['start'] + starts
        df['end'] = df['end'] + ends
        df['seq_nt'] = df['seq_nt'] + seqs_nt
        df['seq_aa'] = df['seq_aa'] + seqs_aa 
        df['strand'] = df['strand'] + [strand]*len(starts)
        df['length_aa'] = df['length_aa'] + lengths_aa
            
        
    return pd.DataFrame(df).drop_duplicates()

In [3]:
#gather files
gb_file = "/lab/solexa_weissman/yhc/viral_smORF/data_downloads/virushostdb/virushostdb.gbff"
genome_df = pd.read_csv("/lab/solexa_weissman/yhc/viral_smORF/data_downloads/virushostdb/virushostdb.human.genome_parsed_filt.csv")
display(genome_df.head())
cds_df = pd.read_csv("/lab/solexa_weissman/yhc/viral_smORF/data_downloads/virushostdb/virushostdb.human.matpro_parsed_filt.csv")
display(cds_df.head())


,virus_name,accession,virus_tax_id,description,num_matpro,num_cds,refseq_id_seg,segmented,genome_type,genome_composition,...,host_name,host_lineage,pmid,evidence,sample_type,source_organism,refseq_id_y,comment,genome_len,genome_seq
0,Human parvovirus B19,NC_000883.2,10798,"Human parvovirus B19, complete genome",6.0,6.0,NC_000883,False,ssDNA,Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,UniProt,NaN,NaN,NC_000883,VALIDATED REFSEQ: This record has undergone va...,5596,CCAAATCAGATGCCGCCGGTCGCCGCCGGTAGGCGGGACTTCCGGT...
1,Human betaherpesvirus 6B,NC_000898.1,32604,"Human herpesvirus 6B, complete genome",104.0,104.0,NC_000898,False,dsDNA,Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,"NCBI Virus, RefSeq, UniProt",NaN,NaN,NC_000898,PROVISIONAL REFSEQ: This record has not yet be...,162114,TCCTCGCGTTTCAAAAATTACTTTAAACTCCCCGGGGGGGTTAAAA...
2,Murray Valley encephalitis virus,NC_000943.1,11079,"Murray Valley encephalitis virus, complete genome",15.0,2.0,NC_000943,False,ssRNA(+),Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,UniProt,NaN,NaN,NC_000943,REVIEWED REFSEQ: This record has been curated ...,11014,AGACGTTCATCTGCGTGAGCTTCCGATCTCAGTATTGTTTGGAAGG...
3,Human alphaherpesvirus 3,NC_001348.1,10335,"Human herpesvirus 3, complete genome",73.0,73.0,NC_001348,False,dsDNA,Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,"NCBI Virus, RefSeq, UniProt",NaN,NaN,NC_001348,VALIDATED REFSEQ: This record has undergone va...,124884,AGGCCAGCCCTCTCGCGGCCCCCTCGAGAGAGAAAAAAAAAAGCGA...
4,Alphapapillomavirus 4,NC_001352.1,337043,"Human papillomavirus - 2, complete genome",7.0,7.0,NC_001352,False,dsDNA,Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,1964523,Literature,NaN,NaN,NC_001352,VALIDATED REFSEQ: This record has undergone va...,7860,ATAATGTATAACTATAATCCTTTATTTAAAAATAGGGTGTGACCGA...


,virus_name,accession,genome_type,virus_tax_id,type,location,strand,gene_id,polypep?,length_aa,...,experiment,old_locus_tag,exception,ribosomal_slippage,citation,standard_name,gene_synonym,number,inference,trans_splicing
0,SARS coronavirus Tor2,NC_004718.3,ssRNA(+),227984,mat_peptide,"join{[13371:13392](+), [13391:13394](+)}",1,NaN,NaN,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Severe acute respiratory syndrome coronavirus 2,NC_045512.2,ssRNA(+),2697049,mat_peptide,[13441:13480](+),1,NaN,NaN,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SARS coronavirus Tor2,NC_004718.3,ssRNA(+),227984,mat_peptide,[13371:13410](+),1,NaN,NaN,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Betacoronavirus England 1,NC_038294.1,ssRNA(+),1263720,mat_peptide,[13408:13450](+),1,NaN,NaN,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Human coronavirus OC43,NC_006213.1,ssRNA(+),31631,mat_peptide,[13316:13358](+),1,NaN,NaN,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
orf_df = []


for j,acc in enumerate(genome_df.accession):
    

    ###################### Compiling ORF DataFrame
    #retrieve descriptors about the genome that the ORFs came from
    #retrieve all ORFs
    sub = genome_df.iloc[j]
    all_orfs = find_orfs_with_trans(Seq.Seq(sub.genome_seq)) 
    all_orfs.insert(1, 'taxid', [sub.virus_tax_id]*len(all_orfs))
    all_orfs.insert(0, 'virus_name', [sub['virus_name']]*len(all_orfs))
    all_orfs.insert(2, 'accession', [sub.accession]*len(all_orfs))
    all_orfs.insert(3, 'orf_name', ['ORF_'+str(i+1) for i in range(len(all_orfs))])
    #all_orfs.insert(3, 'gene_len', [len(all_orfs.iloc[i]['seq_nt']) for i in range(len(all_orfs))])
    orf_df.append(all_orfs)

#filter out non-human host viruses
orf_df = pd.concat(orf_df).sort_values('length_aa').reset_index(drop=True)
display(orf_df.head(10))


# Export as csv
orf_df.to_csv('virushostdb_orf_df_raw.csv')



,virus_name,start,accession,orf_name,taxid,end,strand,seq_nt,seq_aa,length_aa
0,Severe acute respiratory syndrome coronavirus 2,9402,NC_045512.2,ORF_1232,2697049,9375,-1,ATGCTGATATGTCCAAAGCACCAATAG,MLICPKHQ,8
1,Variola virus,164908,NC_001611.1,ORF_3813,10255,164881,-1,CTGCTCTTGTTGTTCACAATATACTAA,MLLLFTIY,8
2,adeno-associated virus 2,3504,NC_001401.2,ORF_164,10804,3477,-1,ATGAGACGGTCCAGACTCTGGCTGTGA,MRRSRLWL,8
3,Cowpox virus,68645,NC_003663.2,ORF_7284,10243,68618,-1,ATGCTGATACATCCTCCACAGACTTGA,MLIHPPQT,8
4,Fort Sherman virus,3830,NC_043615.1,ORF_90,273345,3857,1,ATGTGTCGGATGTCCATCGTGTTTTGA,MCRMSIVF,8
5,Beilong virus,7342,NC_007803.1,ORF_868,341053,7315,-1,CTGGCTTCTCAGGAGCACAATTCTTAG,MASQEHNS,8
6,Beilong virus,7435,NC_007803.1,ORF_865,341053,7408,-1,ATGTCAGAGCGTGCATCATTGTGTTGA,MSERASLC,8
7,Human papillomavirus type 96,1938,NC_005134.2,ORF_59,247269,1965,1,CTGGATTGTTCAGCAAACGATGATTAG,MDCSANDD,8
8,Nairobi sheep disease virus,30,NC_034391.1,ORF_301,194540,3,-1,CTGCTAGTGCCGCAACTATCTCTTTGA,MLVPQLSL,8
9,Fort Sherman virus,4315,NC_043615.1,ORF_101,273345,4342,1,ATGAATATTTGTATCTTCAGGAACTGA,MNICIFRN,8


In [5]:
sum(orf_df.length_aa < 105)

351198

In [6]:
orf_df.seq_aa.nunique()

446089

In [7]:
#merge annotations
orf_df_raw = orf_df.reset_index(drop=True)
orf_df_raw['orf_index'] = orf_df_raw.index
print(len(orf_df))


cds_df['genes_index'] = cds_df.index

print(len(cds_df))
annotated_aa = orf_df_raw.merge(cds_df, on = ['accession','seq_aa']).sort_values('orf_index')
print(len(annotated_aa))
annotated_aa = annotated_aa.loc[annotated_aa.orf_index.drop_duplicates().index].reset_index(drop=True)
annotated_aa = annotated_aa.rename(columns={'virus_name_x':'virus_name', 'strand_x': 'strand', 
                     'seq_nt_x':'seq_nt', 'seq_aa_x':'seq_aa', 'length_aa_x': 'length_aa', 'length_nt_x': 'length_nt'})
annotated_aa['ann'] = [True] * len(annotated_aa)
annotated_aa['matpro_id'] = annotated_aa['protein_id']
print(len(annotated_aa))


cols = ['virus_name','accession', 'matpro_id','product', 'start',  'taxid', 'end',
       'strand', 'seq_nt', 'seq_aa', 'length_aa', 'ann']


orf_df_raw['product'] = orf_df_raw.orf_name
orf_df_raw['ann'] = [False] * len(orf_df_raw)
orf_df_raw['matpro_id'] = ['Unannotated'] * len(orf_df_raw)
orf_df_ann = annotated_aa[cols].copy().reset_index(drop=True)
orf_df_un = orf_df_raw[~orf_df_raw.orf_index.isin(annotated_aa.orf_index)][cols].copy().reset_index(drop=True)

orf_df_annotated=pd.concat([orf_df_ann, orf_df_un]).sort_values(['accession', 'strand', 'start']).sort_values('length_aa').reset_index(drop=True)
#orf_df_annotated=orf_df_annotated.merge(tax_df, on ='accession').sort_values('length_aa').reset_index(drop=True)
display(orf_df_annotated.head(5))
# #orf_df_ann = pd.DataFrame({'matpro_id':['Unannotated']*len(orf_df_raw)})
# #orf_df_ann.loc[annotated_aa.orf_index,'matpro_id'] = annotated_aa.protein_id
# orf_df_annotated=orf_df_raw.copy()
# orf_df_annotated['ann?'] = orf_df_raw.orf_index.isin(annotated_aa.orf_index)
# orf_df_annotated['matpro_id'] = np.array(len(orf_df_raw), dtype = object)
# orf_df_annotated.loc[annotated_aa.orf_index, 'matpro_id'] = annotated_aa['protein_id']
# orf_df_annotated.loc[~orf_df_raw.orf_index.isin(annotated_aa.orf_index), 'matpro_id'] = 'Unannotated'


orf_df_annotated.sort_values(by = 'length_aa').to_csv('virushostdb_orf_df_assigned.csv')



476132
7141
5606
5512


,virus_name,accession,matpro_id,product,start,taxid,end,strand,seq_nt,seq_aa,length_aa,ann
0,Astrovirus MLB3,NC_019028.1,Unannotated,ORF_218,5893,1247114,5866,-1,CTGAATCAATTGCATGGAGAGTTGTGA,MNQLHGEL,8,False
1,Orungo virus,NC_038605.1,Unannotated,ORF_46,1814,40058,1841,1,ATGTCCCAGTCTACCGTGACCACATGA,MSQSTVTT,8,False
2,Molluscum contagiosum virus subtype 1,NC_001731.1,Unannotated,ORF_6301,149386,10280,149359,-1,CTGGCTGGGGTGGTACTTGCGCACTGA,MAGVVLAH,8,False
3,Beilong virus,NC_007803.1,Unannotated,ORF_284,9034,341053,9061,1,CTGGGGGAGCCTATCATCTCCCATTGA,MGEPIISH,8,False
4,Beilong virus,NC_007803.1,Unannotated,ORF_277,8821,341053,8848,1,ATGGATGGGATGCGTGATCATGACTGA,MDGMRDHD,8,False


In [8]:
orf_df_annotated[orf_df_annotated['ann']].reset_index(drop=True).to_csv('virushostdb_orf_df_annotated.csv')



In [9]:
cds_df.sort_values(by = 'length_aa', ascending=True).reset_index(drop=True).to_csv('virushostdb_cds_annotated.csv')
cds_df[cds_df.length_aa < 105].sort_values(by = 'length_aa', ascending=True).reset_index(drop=True).to_csv('virushostdb_cds_annotated_small.csv')




In [10]:
cds_df[cds_df.length_aa < 105].sort_values(by = 'length_aa', ascending=True)

,virus_name,accession,genome_type,virus_tax_id,type,location,strand,gene_id,polypep?,length_aa,...,old_locus_tag,exception,ribosomal_slippage,citation,standard_name,gene_synonym,number,inference,trans_splicing,genes_index
0,SARS coronavirus Tor2,NC_004718.3,ssRNA(+),227984,mat_peptide,"join{[13371:13392](+), [13391:13394](+)}",1,NaN,NaN,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Severe acute respiratory syndrome coronavirus 2,NC_045512.2,ssRNA(+),2697049,mat_peptide,[13441:13480](+),1,NaN,NaN,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,SARS coronavirus Tor2,NC_004718.3,ssRNA(+),227984,mat_peptide,[13371:13410](+),1,NaN,NaN,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
3,Betacoronavirus England 1,NC_038294.1,ssRNA(+),1263720,mat_peptide,[13408:13450](+),1,NaN,NaN,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
4,Human coronavirus OC43,NC_006213.1,ssRNA(+),31631,mat_peptide,[13316:13358](+),1,NaN,NaN,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
5,Middle East respiratory syndrome-related coron...,NC_019843.3,ssRNA(+),1335626,mat_peptide,[13409:13451](+),1,NaN,NaN,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
6,Human coronavirus HKU1,NC_006577.2,ssRNA(+),290028,mat_peptide,[13576:13618](+),1,NaN,NaN,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
7,Human immunodeficiency virus 1,NC_001802.1,ssRNA-RT,11676,mat_peptide,[1424:1466](+),1,NaN,NaN,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
8,Human immunodeficiency virus 1,NC_001802.1,ssRNA-RT,11676,mat_peptide,[1631:1679](+),1,NaN,NaN,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
12,Cosavirus E,NC_012798.1,ssRNA(+),2003651,mat_peptide,[4425:4482](+),1,NaN,NaN,19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12
